# AU-FISH Multi-Seed Pairwise WBF Run

**Protocol:** `time_based`  
**Pair:** `rtdetr_l_faster_rcnn`  
**Members:** `rtdetr_l` + `faster_rcnn`  
**Seed:** `2026`  
**Experiment tag:** `multiseed_wbf_fullset_dedup_v1`

Notebook ini hanya menjalankan fusion untuk satu pair dan satu seed.

Notebook tidak melakukan training atau detector inference. Inputnya adalah raw
validation/test predictions dari dua notebook individual dengan seed yang sama.
WBF dituning hanya pada validation set, lalu konfigurasi terkunci diterapkan satu
kali pada test set.

Seluruh output disimpan pada folder pair dan seed yang spesifik.

In [1]:
# CELL 1 — INSTALL DEPENDENCIES
!pip -q install pycocotools ensemble-boxes pandas

In [2]:
# CELL 2 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# CELL 3 — FIXED PAIRWISE RUN CONFIGURATION
import contextlib
import hashlib
import io
import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from ensemble_boxes import (
    weighted_boxes_fusion,
)
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

PROTOCOL = "time_based"
PAIR_NAME = "rtdetr_l_faster_rcnn"
MODEL_NAMES = ['rtdetr_l', 'faster_rcnn']
SEED = 2026

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/au-fish/dataset_v2"
)
DATASET_ROOT = PROJECT_ROOT / PROTOCOL

INDIVIDUAL_TAG = "multiseed_fullset_dedup_v1"
WBF_TAG = "multiseed_wbf_fullset_dedup_v1"

INDIVIDUAL_ROOT = (
    DATASET_ROOT
    / "experiment_logs"
    / INDIVIDUAL_TAG
)

SHARED_GT_DIR = (
    INDIVIDUAL_ROOT
    / "shared_ground_truth"
)

WBF_ROOT = (
    DATASET_ROOT
    / "experiment_logs"
    / WBF_TAG
)

RUN_ROOT = (
    WBF_ROOT
    / PAIR_NAME
    / f"seed_{SEED}"
)

RAW_DIR = RUN_ROOT / "raw_predictions"
RESULTS_DIR = RUN_ROOT / "results"
MANIFEST_DIR = RUN_ROOT / "manifests"

for path in [
    RAW_DIR,
    RESULTS_DIR,
    MANIFEST_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

STAGE1_IOU_THRESHOLDS = [
    0.45,
    0.50,
    0.55,
    0.60,
]

STAGE1_SKIP_THRESHOLDS = [
    0.001,
    0.01,
    0.03,
]

STAGE1_CONF_TYPES = [
    "avg",
    "box_and_model_avg",
    "max",
]

STAGE2_GLOBAL_WEIGHTS = [
    0.75,
    1.00,
    1.25,
    1.50,
]

print("=" * 80)
print("AU-FISH MULTI-SEED PAIRWISE WBF")
print("Protocol :", PROTOCOL)
print("Pair     :", PAIR_NAME)
print("Models   :", MODEL_NAMES)
print("Seed     :", SEED)
print("Run root :", RUN_ROOT)
print("=" * 80)

AU-FISH MULTI-SEED PAIRWISE WBF
Protocol : time_based
Pair     : rtdetr_l_faster_rcnn
Models   : ['rtdetr_l', 'faster_rcnn']
Seed     : 2026
Run root : /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_wbf_fullset_dedup_v1/rtdetr_l_faster_rcnn/seed_2026


In [4]:
# CELL 4 — FILE HELPERS AND INPUT PREFLIGHT
def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary.replace(path)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def individual_run_root(model_name):
    return (
        INDIVIDUAL_ROOT
        / "individual"
        / model_name
        / f"seed_{SEED}"
    )


def individual_raw_path(
    model_name,
    split_label,
):
    return (
        individual_run_root(model_name)
        / "raw_predictions"
        / f"{split_label}_{model_name}.json"
    )


def individual_results_path(model_name):
    return (
        individual_run_root(model_name)
        / "results"
        / "common_cocoeval_results.csv"
    )


def gt_path_for(split_label):
    file_name = (
        "val_gt_filtered.json"
        if split_label == "validation"
        else "test_gt_filtered.json"
    )

    return SHARED_GT_DIR / file_name


required_inputs = []

for model_name in MODEL_NAMES:
    required_inputs.extend([
        individual_raw_path(
            model_name,
            "validation",
        ),
        individual_raw_path(
            model_name,
            "test",
        ),
        individual_results_path(
            model_name
        ),
    ])

required_inputs.extend([
    gt_path_for("validation"),
    gt_path_for("test"),
])

missing = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Pairwise input files are incomplete. "
        "Run the matching individual notebooks first:\n"
        + "\n".join(missing)
    )

print("✅ All pairwise inputs found.")

✅ All pairwise inputs found.


In [5]:
# CELL 5 — NORMALIZATION AND COVERAGE VALIDATION
def normalize_prediction_dict(predictions):
    normalized = {}

    iterable = (
        predictions.values()
        if isinstance(predictions, dict)
        else predictions
    )

    for record in iterable:
        image_id = int(
            record.get(
                "image_id",
                record.get("id"),
            )
        )

        boxes = [
            [float(value) for value in box]
            for box in record.get(
                "boxes",
                [],
            )
        ]
        scores = [
            float(value)
            for value in record.get(
                "scores",
                [],
            )
        ]
        labels = [
            int(value)
            for value in record.get(
                "labels",
                [],
            )
        ]

        if not (
            len(boxes)
            == len(scores)
            == len(labels)
        ):
            raise RuntimeError(
                f"Length mismatch at image "
                f"{image_id}"
            )

        normalized[str(image_id)] = {
            "image_id": image_id,
            "file_name": str(
                record["file_name"]
            ),
            "width": int(
                record["width"]
            ),
            "height": int(
                record["height"]
            ),
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
        }

    return normalized


def load_prediction_sets(split_label):
    prediction_sets = {}

    for model_name in MODEL_NAMES:
        path = individual_raw_path(
            model_name,
            split_label,
        )

        prediction_sets[model_name] = (
            normalize_prediction_dict(
                load_json(path)
            )
        )

    reference_ids = set(
        prediction_sets[
            MODEL_NAMES[0]
        ].keys()
    )

    for model_name in MODEL_NAMES[1:]:
        current_ids = set(
            prediction_sets[
                model_name
            ].keys()
        )

        if current_ids != reference_ids:
            raise RuntimeError(
                f"Image coverage differs: "
                f"{MODEL_NAMES[0]} vs "
                f"{model_name}; "
                f"missing="
                f"{len(reference_ids-current_ids)}, "
                f"extra="
                f"{len(current_ids-reference_ids)}"
            )

    return prediction_sets

In [6]:
# CELL 6 — WBF AND COMMON COCOEVAL HELPERS
def blank_record(source_record):
    return {
        "image_id": int(
            source_record["image_id"]
        ),
        "file_name": str(
            source_record["file_name"]
        ),
        "width": int(
            source_record["width"]
        ),
        "height": int(
            source_record["height"]
        ),
        "boxes": [],
        "scores": [],
        "labels": [],
    }


def fuse_one_image(
    records,
    weights,
    iou_thr,
    skip_box_thr,
    conf_type,
):
    width = float(records[0]["width"])
    height = float(records[0]["height"])

    boxes_list = []
    scores_list = []
    labels_list = []

    for record in records:
        normalized_boxes = []
        valid_scores = []
        valid_labels = []

        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(
                float,
                box,
            )

            x1 = min(
                max(x1, 0.0),
                width,
            )
            y1 = min(
                max(y1, 0.0),
                height,
            )
            x2 = min(
                max(x2, 0.0),
                width,
            )
            y2 = min(
                max(y2, 0.0),
                height,
            )

            if x2 <= x1 or y2 <= y1:
                continue

            normalized_boxes.append([
                x1 / width,
                y1 / height,
                x2 / width,
                y2 / height,
            ])
            valid_scores.append(
                float(score)
            )
            valid_labels.append(
                int(label)
            )

        boxes_list.append(
            normalized_boxes
        )
        scores_list.append(
            valid_scores
        )
        labels_list.append(
            valid_labels
        )

    fused_record = blank_record(
        records[0]
    )

    if not any(
        len(boxes)
        for boxes in boxes_list
    ):
        return fused_record

    (
        fused_boxes,
        fused_scores,
        fused_labels,
    ) = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        weights=[
            float(value)
            for value in weights
        ],
        iou_thr=float(iou_thr),
        skip_box_thr=float(
            skip_box_thr
        ),
        conf_type=str(conf_type),
        allows_overflow=False,
    )

    for box, score, label in zip(
        fused_boxes,
        fused_scores,
        fused_labels,
    ):
        x1, y1, x2, y2 = map(
            float,
            box,
        )

        fused_record["boxes"].append([
            x1 * width,
            y1 * height,
            x2 * width,
            y2 * height,
        ])
        fused_record["scores"].append(
            float(score)
        )
        fused_record["labels"].append(
            int(label)
        )

    return fused_record


def fuse_prediction_sets(
    prediction_sets,
    parameters,
):
    image_ids = sorted(
        prediction_sets[
            MODEL_NAMES[0]
        ].keys(),
        key=int,
    )

    fused = {}

    for image_id in image_ids:
        records = [
            prediction_sets[
                model_name
            ][image_id]
            for model_name in MODEL_NAMES
        ]

        fused[image_id] = (
            fuse_one_image(
                records=records,
                weights=parameters[
                    "weights"
                ],
                iou_thr=parameters[
                    "iou_thr"
                ],
                skip_box_thr=parameters[
                    "skip_box_thr"
                ],
                conf_type=parameters[
                    "conf_type"
                ],
            )
        )

    return fused


def raw_to_coco_rows(raw):
    rows = []

    for record in raw.values():
        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(
                float,
                box,
            )

            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                continue

            rows.append({
                "image_id": int(
                    record["image_id"]
                ),
                "category_id": int(label) + 1,
                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],
                "score": float(score),
            })

    return rows


def evaluate_common_coco(gt_path, raw):
    rows = raw_to_coco_rows(raw)

    if not rows:
        return {
            "mAP50_95": 0.0,
            "mAP50": 0.0,
            "mAP75": 0.0,
            "AP_coral": 0.0,
            "AP_fish": 0.0,
            "worst_class_AP": 0.0,
            "detections": 0,
        }

    with contextlib.redirect_stdout(
        io.StringIO()
    ):
        coco_gt = COCO(str(gt_path))
        coco_dt = coco_gt.loadRes(rows)

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox",
        )
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    class_ap = {}

    for category_id, class_name in [
        (1, "coral"),
        (2, "fish"),
    ]:
        with contextlib.redirect_stdout(
            io.StringIO()
        ):
            class_eval = COCOeval(
                coco_gt,
                coco_dt,
                "bbox",
            )
            class_eval.params.catIds = [
                category_id
            ]
            class_eval.evaluate()
            class_eval.accumulate()
            class_eval.summarize()

        class_ap[class_name] = float(
            class_eval.stats[0]
        )

    return {
        "mAP50_95": float(
            evaluator.stats[0]
        ),
        "mAP50": float(
            evaluator.stats[1]
        ),
        "mAP75": float(
            evaluator.stats[2]
        ),
        "AP_coral": class_ap["coral"],
        "AP_fish": class_ap["fish"],
        "worst_class_AP": min(
            class_ap["coral"],
            class_ap["fish"],
        ),
        "detections": len(rows),
    }


def rank_key(metrics):
    return (
        float(metrics["mAP50_95"]),
        float(metrics["worst_class_AP"]),
        float(metrics["mAP50"]),
    )

In [7]:
# CELL 7 — STAGE 1 VALIDATION SEARCH
validation_predictions = (
    load_prediction_sets(
        "validation"
    )
)

tuning_rows = []
best_stage1 = None

for (
    iou_thr,
    skip_box_thr,
    conf_type,
) in product(
    STAGE1_IOU_THRESHOLDS,
    STAGE1_SKIP_THRESHOLDS,
    STAGE1_CONF_TYPES,
):
    parameters = {
        "iou_thr": iou_thr,
        "skip_box_thr": skip_box_thr,
        "conf_type": conf_type,
        "weights": [1.0, 1.0],
    }

    fused = fuse_prediction_sets(
        validation_predictions,
        parameters,
    )

    metrics = evaluate_common_coco(
        gt_path_for("validation"),
        fused,
    )

    row = {
        "stage": 1,
        "protocol": PROTOCOL,
        "pair": PAIR_NAME,
        "seed": SEED,
        **parameters,
        **metrics,
    }

    tuning_rows.append(row)

    if (
        best_stage1 is None
        or rank_key(metrics)
        > rank_key(
            best_stage1["metrics"]
        )
    ):
        best_stage1 = {
            "parameters": parameters,
            "metrics": metrics,
        }

print("Best Stage 1:")
print(best_stage1)

Best Stage 1:
{'parameters': {'iou_thr': 0.6, 'skip_box_thr': 0.001, 'conf_type': 'max', 'weights': [1.0, 1.0]}, 'metrics': {'mAP50_95': 0.41833755869964534, 'mAP50': 0.6600484000257786, 'mAP75': 0.4577152666125991, 'AP_coral': 0.4792974056850834, 'AP_fish': 0.3573777117142074, 'worst_class_AP': 0.3573777117142074, 'detections': 42077}}


In [8]:
# CELL 8 — STAGE 2 GLOBAL PAIR-WEIGHT SEARCH
best_final = None

for weight_1, weight_2 in product(
    STAGE2_GLOBAL_WEIGHTS,
    STAGE2_GLOBAL_WEIGHTS,
):
    parameters = {
        "iou_thr": (
            best_stage1[
                "parameters"
            ]["iou_thr"]
        ),
        "skip_box_thr": (
            best_stage1[
                "parameters"
            ]["skip_box_thr"]
        ),
        "conf_type": (
            best_stage1[
                "parameters"
            ]["conf_type"]
        ),
        "weights": [
            weight_1,
            weight_2,
        ],
    }

    fused = fuse_prediction_sets(
        validation_predictions,
        parameters,
    )

    metrics = evaluate_common_coco(
        gt_path_for("validation"),
        fused,
    )

    row = {
        "stage": 2,
        "protocol": PROTOCOL,
        "pair": PAIR_NAME,
        "seed": SEED,
        **parameters,
        **metrics,
    }

    tuning_rows.append(row)

    if (
        best_final is None
        or rank_key(metrics)
        > rank_key(
            best_final["metrics"]
        )
    ):
        best_final = {
            "parameters": parameters,
            "metrics": metrics,
        }

best_parameters = {
    "protocol": PROTOCOL,
    "pair": PAIR_NAME,
    "model_order": MODEL_NAMES,
    "seed": SEED,
    "selection_split": "validation",
    "primary_objective": "mAP50-95",
    "tie_break_1": "worst-class AP",
    "tie_break_2": "mAP50",
    **best_final["parameters"],
    "validation_metrics": (
        best_final["metrics"]
    ),
}

tuning_df = pd.DataFrame(
    tuning_rows
)

tuning_df.to_csv(
    RESULTS_DIR
    / "validation_tuning_results.csv",
    index=False,
)

save_json(
    best_parameters,
    RESULTS_DIR
    / "best_wbf_parameters.json",
)

print("✅ WBF parameters locked:")
print(json.dumps(
    best_parameters,
    indent=2,
))

✅ WBF parameters locked:
{
  "protocol": "time_based",
  "pair": "rtdetr_l_faster_rcnn",
  "model_order": [
    "rtdetr_l",
    "faster_rcnn"
  ],
  "seed": 2026,
  "selection_split": "validation",
  "primary_objective": "mAP50-95",
  "tie_break_1": "worst-class AP",
  "tie_break_2": "mAP50",
  "iou_thr": 0.6,
  "skip_box_thr": 0.001,
  "conf_type": "max",
  "weights": [
    1.5,
    0.75
  ],
  "validation_metrics": {
    "mAP50_95": 0.49803749931952973,
    "mAP50": 0.7540163256265153,
    "mAP75": 0.5460856002864312,
    "AP_coral": 0.5728987134653738,
    "AP_fish": 0.4231762851736859,
    "worst_class_AP": 0.4231762851736859,
    "detections": 42029
  }
}


In [9]:
# CELL 9 — LOCKED VALIDATION AND ONE-TIME TEST EVALUATION
final_rows = []

for split_label in [
    "validation",
    "test",
]:
    prediction_sets = (
        validation_predictions
        if split_label == "validation"
        else load_prediction_sets(
            "test"
        )
    )

    fused = fuse_prediction_sets(
        prediction_sets,
        best_parameters,
    )

    fused_path = (
        RAW_DIR
        / f"{split_label}_wbf_fused.json"
    )

    save_json(
        fused,
        fused_path,
    )

    metrics = evaluate_common_coco(
        gt_path_for(split_label),
        fused,
    )

    final_rows.append({
        "protocol": PROTOCOL,
        "pair": PAIR_NAME,
        "seed": SEED,
        "split": split_label,
        **metrics,
    })

final_df = pd.DataFrame(
    final_rows
)

final_df.to_csv(
    RESULTS_DIR
    / "final_ensemble_results.csv",
    index=False,
)

display(final_df)

,protocol,pair,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections
0,time_based,rtdetr_l_faster_rcnn,2026,validation,0.498037,0.754016,0.546086,0.572899,0.423176,0.423176,42029
1,time_based,rtdetr_l_faster_rcnn,2026,test,0.451312,0.698565,0.481622,0.520132,0.382492,0.382492,58248


In [10]:
# CELL 10 — INDIVIDUAL RESULTS AND PAIRED ENSEMBLE GAIN
individual_rows = []

for model_name in MODEL_NAMES:
    result_df = pd.read_csv(
        individual_results_path(
            model_name
        )
    )

    for split_label in [
        "validation",
        "test",
    ]:
        row = result_df[
            result_df["split"]
            == split_label
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"{model_name}/{split_label}: "
                f"expected one result row, "
                f"found {len(row)}"
            )

        record = row.iloc[0].to_dict()
        record["pair"] = PAIR_NAME

        individual_rows.append(record)

individual_df = pd.DataFrame(
    individual_rows
)

individual_df.to_csv(
    RESULTS_DIR
    / "individual_results.csv",
    index=False,
)

test_individual = individual_df[
    individual_df["split"] == "test"
]

strongest_row = test_individual.loc[
    test_individual[
        "mAP50_95"
    ].idxmax()
]

ensemble_test = final_df[
    final_df["split"] == "test"
].iloc[0]

absolute_gain = float(
    ensemble_test["mAP50_95"]
    - strongest_row["mAP50_95"]
)

relative_gain_percent = (
    100.0
    * absolute_gain
    / float(
        strongest_row["mAP50_95"]
    )
    if strongest_row["mAP50_95"] > 0
    else np.nan
)

gain_df = pd.DataFrame([
    {
        "protocol": PROTOCOL,
        "pair": PAIR_NAME,
        "seed": SEED,
        "strongest_member": (
            strongest_row["model"]
        ),
        "strongest_member_mAP50_95": (
            strongest_row[
                "mAP50_95"
            ]
        ),
        "ensemble_mAP50_95": (
            ensemble_test[
                "mAP50_95"
            ]
        ),
        "absolute_gain": absolute_gain,
        "relative_gain_percent": (
            relative_gain_percent
        ),
        "improved": absolute_gain > 0,
    }
])

gain_df.to_csv(
    RESULTS_DIR
    / "ensemble_gain.csv",
    index=False,
)

display(individual_df)
display(gain_df)

,protocol,model,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections,pair,selected_epoch
0,time_based,rtdetr_l,2026,validation,0.509871,0.743914,0.550802,0.599059,0.420682,0.420682,60900,rtdetr_l_faster_rcnn,NaN
1,time_based,rtdetr_l,2026,test,0.459451,0.692276,0.476728,0.537110,0.381791,0.381791,84600,rtdetr_l_faster_rcnn,NaN
2,time_based,faster_rcnn,2026,validation,0.272865,0.513951,0.273485,0.306386,0.239344,0.239344,7237,rtdetr_l_faster_rcnn,1.0
3,time_based,faster_rcnn,2026,test,0.248974,0.497525,0.230875,0.268220,0.229728,0.229728,10395,rtdetr_l_faster_rcnn,1.0


,protocol,pair,seed,strongest_member,strongest_member_mAP50_95,ensemble_mAP50_95,absolute_gain,relative_gain_percent,improved
0,time_based,rtdetr_l_faster_rcnn,2026,rtdetr_l,0.459451,0.451312,-0.008139,-1.771466,False


In [11]:
# CELL 11 — PAIR-SPECIFIC SNAPSHOT MANIFEST
manifest = {
    "protocol": PROTOCOL,
    "pair": PAIR_NAME,
    "model_order": MODEL_NAMES,
    "seed": SEED,
    "individual_experiment_tag": (
        INDIVIDUAL_TAG
    ),
    "wbf_experiment_tag": WBF_TAG,
    "validation_fused": str(
        RAW_DIR
        / "validation_wbf_fused.json"
    ),
    "test_fused": str(
        RAW_DIR
        / "test_wbf_fused.json"
    ),
    "best_wbf_parameters": str(
        RESULTS_DIR
        / "best_wbf_parameters.json"
    ),
    "validation_tuning_results": str(
        RESULTS_DIR
        / "validation_tuning_results.csv"
    ),
    "final_ensemble_results": str(
        RESULTS_DIR
        / "final_ensemble_results.csv"
    ),
    "individual_results": str(
        RESULTS_DIR
        / "individual_results.csv"
    ),
    "ensemble_gain": str(
        RESULTS_DIR
        / "ensemble_gain.csv"
    ),
    "input_files": {
        model_name: {
            "validation": str(
                individual_raw_path(
                    model_name,
                    "validation",
                )
            ),
            "test": str(
                individual_raw_path(
                    model_name,
                    "test",
                )
            ),
        }
        for model_name in MODEL_NAMES
    },
}

manifest["checksums"] = {}

for label in [
    "validation_fused",
    "test_fused",
    "best_wbf_parameters",
    "validation_tuning_results",
    "final_ensemble_results",
    "individual_results",
    "ensemble_gain",
]:
    path = Path(manifest[label])

    if not path.exists():
        raise FileNotFoundError(
            f"Missing output: {path}"
        )

    manifest["checksums"][
        label
    ] = sha256_file(path)

save_json(
    manifest,
    MANIFEST_DIR
    / "pair_snapshot_manifest.json",
)

print("✅ Pairwise multi-seed run complete.")
print("Run root:", RUN_ROOT)
print("Fused outputs are isolated by protocol/pair/seed.")

✅ Pairwise multi-seed run complete.
Run root: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_wbf_fullset_dedup_v1/rtdetr_l_faster_rcnn/seed_2026
Fused outputs are isolated by protocol/pair/seed.


## Output structure

```text
{protocol}/experiment_logs/multiseed_wbf_fullset_dedup_v1/
└── {pair}/seed_{seed}/
    ├── raw_predictions/
    │   ├── validation_wbf_fused.json
    │   └── test_wbf_fused.json
    ├── results/
    │   ├── validation_tuning_results.csv
    │   ├── best_wbf_parameters.json
    │   ├── final_ensemble_results.csv
    │   ├── individual_results.csv
    │   └── ensemble_gain.csv
    └── manifests/
        └── pair_snapshot_manifest.json
```

Tidak ada fused prediction generik bersama. Setiap snapshot berada di folder
pair dan seed-nya sendiri.